In [1]:
#!/usr/bin/env python
import os
import numpy as np
import networkx as nx
import MDAnalysis as mda
import pandas as pd
from math import exp
from scipy.spatial import ConvexHull, cKDTree
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

# OpenBabel imports for ligand and minimal protein conversion.
from openbabel import openbabel as ob
from openbabel import pybel

# ---------------------------
# Modern CGAL SWIG Bindings (from https://github.com/CGAL/cgal-swig-bindings)
# ---------------------------
from CGAL.CGAL_Kernel import Point_3, Weighted_point_3
from CGAL.CGAL_Triangulation_3 import Regular_triangulation_3

# ---------------------------
# Protein Ingestion and Pocket Functions
# ---------------------------
def convert_to_pdbqt_protein(input_file, output_file=None):
    if os.path.splitext(input_file)[1].lower() != ".pdb":
        raise ValueError("Protein input must be a PDB file.")
    if output_file is None:
        base = os.path.splitext(input_file)[0]
        output_file = base + "_rigid.pdbqt"
    conv = ob.OBConversion()
    conv.SetInFormat("pdb")
    conv.SetOutFormat("pdbqt")
    conv.AddOption("r", ob.OBConversion.OUTOPTIONS)  # rigid output
    obmol = ob.OBMol()
    conv.ReadFile(obmol, input_file)
    obmol.CorrectForPH()  # adjust protonation if needed
    obmol.AddHydrogens()  # ensure hydrogens are present
    conv.WriteFile(obmol, output_file)
    return output_file

def convert_to_pdbqt_ligand(input_file, output_file=None):
    ext = os.path.splitext(input_file)[1].lower()[1:]
    if output_file is None:
        base = os.path.splitext(input_file)[0]
        output_file = base + ".pdbqt"
    mols = list(pybel.readfile(ext, input_file))
    if not mols:
        raise ValueError(f"No molecules found in {input_file}")
    mol = mols[0]
    mol.addh()       # Add hydrogens (for pH ~7.4)
    mol.make3D()     # Generate 3D geometry
    mol.calccharges(model='gasteiger')
    mol.write("pdbqt", output_file, overwrite=True)
    return output_file

def load_protein(pdb_file, trajectory_file=None):
    if trajectory_file is not None:
        u = mda.Universe(pdb_file, trajectory_file)
    else:
        u = mda.Universe(pdb_file)
    return u

def get_protein_only(u):
    return u.select_atoms("protein")

def get_vdw_radius(atom):
    """Return the real van der Waals radius (in Å) for an atom."""
    VDW = {'H': 1.2, 'C': 1.7, 'N': 1.55, 'O': 1.52, 'S': 1.8}
    try:
        elem = atom.element.strip().upper()
    except AttributeError:
        elem = atom.name[0].upper()
    return VDW.get(elem, 1.7)

def compute_clearance(point, protein, k=5):
    """
    Compute clearance at a 3D point as:
       clearance = min_{atom in protein} (||point - atom.pos|| - vdw_radius(atom))
    Uses a cKDTree for efficiency.
    """
    positions = protein.positions
    radii = np.array([get_vdw_radius(a) for a in protein])
    tree = cKDTree(positions)
    dists, idxs = tree.query(point.reshape(1,3), k=k)
    if k == 1:
        dists = dists[:, np.newaxis]
        idxs = idxs[:, np.newaxis]
    return np.min(dists - radii[idxs])

def get_pocket_center(u, method="atom", active_site_atom_sel="resid 100", ligand_sel="", **kwargs):
    """
    Determine a starting point for tunnel search.
    Currently using the "atom" method (returns the position of the first active site atom).
    """
    if method == "atom":
        active_atoms = u.select_atoms(active_site_atom_sel)
        if len(active_atoms) == 0:
            raise ValueError("No active site atom found!")
        return active_atoms[0].position, None
    else:
        raise ValueError("Only 'atom' method is implemented in this example.")

# ---------------------------
# CGAL-based Weighted Voronoi (Power Diagram) Tunnel Searcher
# ---------------------------
# ---------------------------
# CGAL-based Weighted Voronoi (Power Diagram) Tunnel Searcher
# ---------------------------
from CGAL.CGAL_Kernel import Point_3, Weighted_point_3
from CGAL.CGAL_Triangulation_3 import Regular_triangulation_3

def build_voronoi_graph_cgal(protein, noise=1e-3, clearance_threshold=0.0, exponent=2):
    """
    Build a NetworkX graph from a weighted Voronoi (power) diagram using CGAL's Regular_triangulation_3.
    Each protein atom is converted to a weighted point with weight = (vdW radius)^2.
    For each finite cell, the dual (power diagram vertex) is computed.
    Only vertices with clearance > clearance_threshold are kept.
    Edges are added for facets shared by cells, with weight = distance / (min(clearance)^exponent).
    """
    # Build a list of weighted points.
    weighted_points = []
    for atom in protein:
        # Add a small noise and ensure coordinates are native Python floats.
        pos = atom.position + np.random.uniform(-noise, noise, 3)
        p = Point_3(float(pos[0]), float(pos[1]), float(pos[2]))
        r = get_vdw_radius(atom)
        weight = float(r**2)
        wp = Weighted_point_3(p, weight)
        weighted_points.append(wp)
    
    print("grid made")
    
    # Construct the regular triangulation from the list of weighted points.
    rt = Regular_triangulation_3(weighted_points)
    
    # Build vertices from the duals of finite cells.
    vertices = []
    cell_to_vertex = {}
    for cell in rt.finite_cells():
        # Instead of calling cell.dual(), call the triangulation’s dual() method.
        dual = rt.dual(cell)
        if dual is not None:
            v_coord = np.array([float(dual.x()), float(dual.y()), float(dual.z())])
            c_val = compute_clearance(v_coord, protein, k=5)
            if c_val > clearance_threshold:
                idx = len(vertices)
                vertices.append(v_coord)
                cell_to_vertex[cell] = idx

    print("vertexes made")

    # Build the graph.
    G = nx.Graph()
    for i, v in enumerate(vertices):
        c_val = compute_clearance(v, protein, k=5)
        G.add_node(i, coord=v, clearance=c_val, bulk=False)

    for facet in rt.finite_facets():
        # A facet is given as (cell, i) where i is the index of the vertex opposite the facet.
        cell, i = facet
        neighbor = rt.neighbor(cell, i)
        if cell in cell_to_vertex and neighbor in cell_to_vertex:
            u_idx = cell_to_vertex[cell]
            v_idx = cell_to_vertex[neighbor]
            coord_u = G.nodes[u_idx]['coord']
            coord_v = G.nodes[v_idx]['coord']
            d = np.linalg.norm(coord_u - coord_v)
            c_edge = min(G.nodes[u_idx]['clearance'], G.nodes[v_idx]['clearance'])
            edge_weight = d / (c_edge**exponent) if c_edge > 0 else np.inf
            G.add_edge(u_idx, v_idx, distance=d, weight=edge_weight)
    
    print("add facets")

    return G

def mark_bulk_solvent(G, protein, r_probe=3.0):
    coords = protein.positions
    hull = ConvexHull(coords)
    external_nodes = set()
    for n, data in G.nodes(data=True):
        pt = data['coord']
        # Check if point is outside the convex hull.
        if (not np.all(np.dot(hull.equations[:,:-1], pt) + hull.equations[:,-1] <= 1e-12)) or (data['clearance'] >= r_probe):
            external_nodes.add(n)
    bulk = set()
    for n in external_nodes:
        bulk.update(nx.node_connected_component(G, n))
    for n in G.nodes():
        G.nodes[n]['bulk'] = (n in bulk)
    return G

def find_start_node(G, starting_point):
    sp = np.array(starting_point)
    min_d = np.inf
    start_node = None
    for n, data in G.nodes(data=True):
        if data.get('bulk', False):
            continue
        d = np.linalg.norm(data['coord'] - sp)
        if d < min_d:
            min_d = d
            start_node = n
    if start_node is None:
        raise ValueError("No non–bulk node found near the starting point.")
    return start_node

def find_tunnel_paths(G, start_node):
    lengths, paths = nx.single_source_dijkstra(G, start_node, weight="weight")
    candidates = []
    for node, d in lengths.items():
        if G.nodes[node].get('bulk', False):
            candidates.append((paths[node], d))
    return candidates

def analyze_tunnel(path, G, protein, probe_margin=1.0):
    coords = [G.nodes[n]['coord'] for n in path]
    clearances = [G.nodes[n]['clearance'] for n in path]
    length = sum(np.linalg.norm(np.array(coords[i]) - np.array(coords[i+1]))
                 for i in range(len(coords)-1))
    cost = sum(G.edges[u, v]['weight'] for u, v in zip(path[:-1], path[1:]))
    bottleneck_radius = min(clearances)
    bottleneck_idx = np.argmin(clearances)
    bottleneck_coord = coords[bottleneck_idx]
    sel_str = "point {} {} {} distance {}".format(bottleneck_coord[0],
                                                  bottleneck_coord[1],
                                                  bottleneck_coord[2],
                                                  bottleneck_radius + probe_margin)
    close_atoms = protein.select_atoms(sel_str)
    bottleneck_residues = ",".join(sorted({f"{res.resname}{res.resid}" for res in close_atoms.residues}))
    return {
        "length": length,
        "cost": cost,
        "bottleneck_radius": bottleneck_radius,
        "bottleneck_coord": bottleneck_coord,
        "bottleneck_residues": bottleneck_residues,
        "throughput": exp(-cost)
    }

def save_tunnel_as_pdb(path, G, filename, chain_id="T"):
    pdb_lines = []
    for i, n in enumerate(path, start=1):
        coord = G.nodes[n]['coord']
        clearance = G.nodes[n]['clearance']
        line = ("ATOM  {:5d}  CA  TUN {}{:4d}    {:8.3f}{:8.3f}{:8.3f}"
                "  1.00{:6.2f}\n").format(i, chain_id, 1, coord[0], coord[1], coord[2], clearance)
        pdb_lines.append(line)
    with open(filename, "w") as f:
        f.writelines(pdb_lines)

# ---------------------------
# Clustering routines (tunnel centerline sampling and hierarchical clustering)
# ---------------------------
def sample_tunnel(path, G, n_samples=10):
    coords = np.array([G.nodes[n]['coord'] for n in path])
    dists = np.linalg.norm(np.diff(coords, axis=0), axis=1)
    cumdist = np.concatenate(([0], np.cumsum(dists)))
    total_length = cumdist[-1]
    sample_d = np.linspace(0, total_length, n_samples)
    sampled = []
    for d in sample_d:
        idx = np.searchsorted(cumdist, d)
        if idx == 0:
            sampled.append(coords[0])
        elif idx >= len(cumdist):
            sampled.append(coords[-1])
        else:
            t = (d - cumdist[idx-1]) / (cumdist[idx] - cumdist[idx-1])
            point = (1 - t)*coords[idx-1] + t*coords[idx]
            sampled.append(point)
    return np.array(sampled)

def tunnel_distance(tunnel1, tunnel2, G, n_samples=10):
    samp1 = sample_tunnel(tunnel1, G, n_samples)
    samp2 = sample_tunnel(tunnel2, G, n_samples)
    return np.mean(np.linalg.norm(samp1 - samp2, axis=1))

def cluster_tunnels(tunnel_paths, G, n_samples=10, cluster_threshold=2.0):
    n = len(tunnel_paths)
    if n == 0:
        return np.array([])
    D = np.zeros((n, n))
    for i in range(n):
        for j in range(i+1, n):
            d = tunnel_distance(tunnel_paths[i], tunnel_paths[j], G, n_samples)
            D[i, j] = d
            D[j, i] = d
    D_cond = squareform(D)
    Z = linkage(D_cond, method='average')
    labels = fcluster(Z, t=cluster_threshold, criterion='distance')
    return labels

def find_tunnels_voronoi(universe, starting_point, protein_sel="protein",
                         r_probe=3.0, exponent=2, noise=1e-3,
                         clearance_threshold=0.0, margin=5.0,
                         cluster_threshold=2.0, n_samples=10, probe_margin=1.0,
                         use_cgal=True):
    protein = universe.select_atoms(protein_sel)
    if use_cgal:
        G = build_voronoi_graph_cgal(protein, noise=noise,
                                     clearance_threshold=clearance_threshold,
                                     exponent=exponent)
    else:
        raise NotImplementedError("Only CGAL-based version is implemented.")
    G = mark_bulk_solvent(G, protein, r_probe=r_probe)
    start_node = find_start_node(G, starting_point)
    candidate_paths = find_tunnel_paths(G, start_node)
    if len(candidate_paths) == 0:
        raise ValueError("No tunnels found from the starting point to bulk solvent.")
    tunnels = []
    tunnel_paths = []
    for i, (path, cost) in enumerate(candidate_paths):
        metrics = analyze_tunnel(path, G, protein, probe_margin=probe_margin)
        pdb_filename = f"tunnel_{i}.pdb"
        save_tunnel_as_pdb(path, G, pdb_filename)
        metrics["tunnel_id"] = i
        metrics["pdb_file"] = pdb_filename
        tunnels.append(metrics)
        tunnel_paths.append(path)
    labels = cluster_tunnels(tunnel_paths, G, n_samples=n_samples, cluster_threshold=cluster_threshold)
    for t, lab in zip(tunnels, labels):
        t["cluster_id"] = lab
    df = pd.DataFrame(tunnels)
    return df

# ---------------------------
# PyCaverDock Interface
# ---------------------------
import pycaverdock.experiment as cdexp

def run_caverdock_on_tunnel(receptor_pdb, tunnel_pdb, ligand_pdbqt,
                             discretization_steps=50,
                             exhaustiveness=8,
                             output_dir="caverdock_results"):
    """
    Run a CaverDock experiment using PyCaverDock.
    
    Parameters:
      - receptor_pdb: Path to receptor PDBQT file.
      - tunnel_pdb: Path to tunnel PDB file.
      - ligand_pdbqt: Path to ligand PDBQT file.
      - discretization_steps: Number of discretization steps along the tunnel.
      - exhaustiveness: Docking exhaustiveness.
      - output_dir: Output directory.
      
    Returns a dictionary with the experiment object, energy profile, and trajectory.
    """
    experiment = cdexp.CaverDockExperiment(
        receptor=receptor_pdb,
        tunnel=tunnel_pdb,
        ligand=ligand_pdbqt,
        discretization_steps=discretization_steps,
        exhaustiveness=exhaustiveness,
        output_dir=output_dir
    )
    experiment.run()
    return {
        "experiment": experiment,
        "energy_profile": experiment.energy_profile,
        "trajectory": experiment.trajectory
    }

# ---------------------------
# Py3Dmol Visualization Functions
# ---------------------------
import py3Dmol

def visualize_protein_and_tunnels(protein_pdb, tunnel_df, sphere_opacity=0.8, sphere_color="lime"):
    """
    Visualize the protein (from a PDB file) and overlay each tunnel as a series of overlapping spheres.
    Each sphere is centered at a tunnel centerline point (read from the tunnel PDB file) with radius equal
    to its clearance (Rmax). Uses py3Dmol.
    """
    view = py3Dmol.view(width=800, height=600)
    with open(protein_pdb, "r") as f:
        pdb_data = f.read()
    view.addModel(pdb_data, "pdb")
    view.setStyle({'cartoon': {'color': 'spectrum'}})
    
    for idx, row in tunnel_df.iterrows():
        tunnel_file = row["pdb_file"]
        with open(tunnel_file, "r") as f:
            tunnel_pdb_data = f.read()
        lines = tunnel_pdb_data.splitlines()
        for line in lines:
            if line.startswith("ATOM"):
                x = float(line[30:38].strip())
                y = float(line[38:46].strip())
                z = float(line[46:54].strip())
                clearance = float(line[60:66].strip())
                view.addSphere({
                    'center': {'x': x, 'y': y, 'z': z},
                    'radius': clearance,
                    'opacity': sphere_opacity,
                    'color': sphere_color
                })
    view.zoomTo()
    return view.show()

# ---------------------------
# Main Integrated Pipeline
# ---------------------------
def main_pipeline(protein_input, ligand_input, active_site_atom_sel,
                  pocket_method="atom", use_cgal=True, **kwargs):
    """
    Integrated pipeline:
      1. Load the protein from the provided PDB file.
      2. Determine the active site starting point using get_pocket_center.
      3. Run the CGAL-based weighted Voronoi tunnel search.
      4. Select one tunnel (here, highest throughput) and run a CaverDock experiment.
      5. Visualize the protein and all identified tunnels.
      
    Assumes:
      - protein_input is a PDB file.
      - ligand_input is a PDBQT file.
      
    Returns a dictionary with:
         - 'tunnel_df': DataFrame of found tunnels.
         - 'docking_results': Results from the CaverDock experiment.
         - 'py3Dmol_view': HTML/JS view for visualization.
    """
    # For docking, assume the receptor and ligand files are already in appropriate format.
    receptor_pdbqt = protein_input
    ligand_pdbqt = convert_to_pdbqt_ligand(ligand_input)
    
    u = load_protein(protein_input)
    protein = get_protein_only(u)
    
    starting_point, _ = get_pocket_center(u, method=pocket_method,
                                          active_site_atom_sel=active_site_atom_sel,
                                          ligand_sel="", **kwargs)
    print("Using starting point:", starting_point)
    
    tunnel_df = find_tunnels_voronoi(u, starting_point, protein_sel="protein",
                                      r_probe=3.0, exponent=2, noise=1e-3,
                                      clearance_threshold=0.0, margin=5.0,
                                      cluster_threshold=2.0, n_samples=10, probe_margin=1.0,
                                      use_cgal=use_cgal)
    print("Found tunnels:")
    print(tunnel_df)
    
    selected_tunnel = tunnel_df.sort_values("throughput", ascending=False).iloc[0]
    tunnel_pdb = selected_tunnel["pdb_file"]
    print("Selected tunnel for docking:", tunnel_pdb)
    
    docking_results = run_caverdock_on_tunnel(receptor_pdbqt, tunnel_pdb, ligand_pdbqt,
                                               discretization_steps=50,
                                               exhaustiveness=8,
                                               output_dir="caverdock_results")
    print("Docking complete. Energy profile:")
    print(docking_results["energy_profile"])
    
    view_html = visualize_protein_and_tunnels(protein_input, tunnel_df)
    
    return {"tunnel_df": tunnel_df, "docking_results": docking_results, "py3Dmol_view": view_html}

# ---------------------------
# Example Usage
# ---------------------------
if __name__ == '__main__':
    # Replace with your actual file paths and active site selection.
    protein_file = "1be0.pdb"   # PDB file for the protein
    ligand_file = "EtOH.sdf"     # ligand file
    active_site_sel = "resid 100"         # Example active site selection
    results = main_pipeline(protein_file, ligand_file, active_site_sel,
                            pocket_method="atom", use_cgal=True)
    # The results dictionary contains:
    #  - a DataFrame of tunnels,
    #  - docking results from PyCaverDock,
    #  - and a py3Dmol view (HTML/JS) of the protein with tunnels.


/home/erikna/miniconda3/envs/miner/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using starting point: [26.426 24.429 43.57 ]
grid made
vertexes made


AttributeError: 'Regular_triangulation_3' object has no attribute 'neighbor'